# TCN Baseline Shared Session

TCN 모델 전용 실험 노트북입니다. 각 실험자는 `SESSION_ID`를 고유하게 지정하고 같은 공유 Drive 폴더에서 독립적으로 학습/시각화/공유할 수 있습니다.

## 1. Drive 마운트와 프로젝트 루트

공유 폴더 경로가 자동 탐색되지 않으면 `PROJECT_ROOT_OVERRIDE`에 직접 입력합니다.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
except Exception:
    print('Not running in Colab or Drive mount skipped.')

PROJECT_ROOT_OVERRIDE = None
PROJECT_ROOT_CANDIDATES = [
    Path('/content/drive/MyDrive/Graduate-Project/Falling-Model-Development'),
    Path('/content/drive/MyDrive/Falling-Model-Development'),
    Path('/content/drive/MyDrive/졸업 과제/Falling-Model-Development'),
    Path('/content/drive/MyDrive/졸업 과제/Falling-Detection-Development'),
    Path.cwd(),
]

if PROJECT_ROOT_OVERRIDE is not None:
    PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE)
else:
    PROJECT_ROOT = next((p for p in PROJECT_ROOT_CANDIDATES if (p / 'scripts' / 'train_baseline.py').exists()), None)
    if PROJECT_ROOT is None:
        raise FileNotFoundError('Project root not found. Set PROJECT_ROOT_OVERRIDE.')

os.chdir(PROJECT_ROOT)
print('PROJECT_ROOT =', PROJECT_ROOT)

## 2. 런타임 준비

초기 Colab 런타임에서 필요한 패키지를 설치합니다.

In [ ]:
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'tensorflow', 'pandas', 'scikit-learn', 'matplotlib'],
    check=False,
)
import pandas as pd
from IPython.display import Image, Markdown, display
print('Runtime ready.')

## 3. 세션 배정

`SESSION_ID`는 공유 폴더 안에서 결과 충돌을 막는 이름입니다. 예: `tcn_01_min`, `tcn_02_jiyun`.

In [ ]:
MODEL_TYPE = 'tcn'
SESSION_ID = 'tcn_01_yourname'
OWNER = 'yourname'
PREPROCESSING = 'filtered'  # raw, filtered, both
SMOKE = True
EXPORT_TFLITE = True

SESSION_OUTPUT_ROOT = PROJECT_ROOT / 'results' / 'shared_sessions' / MODEL_TYPE / SESSION_ID
SESSION_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
manifest = {
    'model_type': MODEL_TYPE,
    'session_id': SESSION_ID,
    'owner': OWNER,
    'preprocessing': PREPROCESSING,
    'smoke': SMOKE,
    'output_root': str(SESSION_OUTPUT_ROOT),
}
(SESSION_OUTPUT_ROOT / 'session_manifest.json').write_text(json.dumps(manifest, indent=2, ensure_ascii=False))
display(pd.DataFrame([manifest]))

## 4. TCN 실험 설정 확인

`filtered`는 `B-TCN-D`, `raw`는 `B-TCN-raw`를 실행합니다. `both`는 둘 다 실행합니다.

In [ ]:
CONFIG_BY_PREPROCESSING = {
    'raw': ['B-TCN-raw'],
    'filtered': ['B-TCN-D'],
    'both': ['B-TCN-raw', 'B-TCN-D'],
}
selected_experiments = CONFIG_BY_PREPROCESSING[PREPROCESSING]
rows = []
for exp_id in selected_experiments:
    config_path = PROJECT_ROOT / 'configs' / 'experiments' / 'phase0' / f'{exp_id}.json'
    rows.append(json.loads(config_path.read_text()))
display(pd.DataFrame(rows)[['experiment_id', 'model_type', 'preprocessing', 'source_csv', 'feature_set', 'data_scope', 'target_steps', 'epochs']])

## 5. 학습 실행

Keras epoch 진행률과 export 로그가 실시간으로 출력됩니다.

In [ ]:
def run_streaming(cmd):
    print(' '.join(str(part) for part in cmd))
    process = subprocess.Popen([str(part) for part in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
    code = process.wait()
    if code != 0:
        raise RuntimeError(f'Command failed: {code}')

for exp_id in selected_experiments:
    display(Markdown(f'### Running `{exp_id}` in session `{SESSION_ID}`'))
    config_path = PROJECT_ROOT / 'configs' / 'experiments' / 'phase0' / f'{exp_id}.json'
    cmd = [sys.executable, 'scripts/train_baseline.py', '--config', config_path, '--project-root', PROJECT_ROOT, '--output-root', SESSION_OUTPUT_ROOT]
    if SMOKE:
        cmd.append('--smoke')
    if not EXPORT_TFLITE:
        cmd.append('--no-export-tflite')
    run_streaming(cmd)

## 6. 결과 테이블

마지막 실행 실험의 split별 성능과 threshold sweep을 확인합니다.

In [ ]:
result_dir = SESSION_OUTPUT_ROOT / selected_experiments[-1]
metrics = json.loads((result_dir / 'metrics.json').read_text())
summary_rows = []
for key, item in metrics['metrics'].items():
    summary_rows.append({
        'split_runtime': key,
        'accuracy': item.get('accuracy'),
        'precision': item.get('precision'),
        'recall': item.get('recall'),
        'f1': item.get('f1'),
        'auc_roc': item.get('auc_roc'),
        'pr_auc': item.get('pr_auc'),
    })
display(pd.DataFrame(summary_rows))
display(Markdown(f"Selected threshold: `{metrics['threshold_selection']['threshold']:.3f}`"))
display(pd.DataFrame(metrics['threshold_selection']['sweep']))

## 7. 결과 시각화

학습 곡선, threshold sweep, window 분포, confusion matrix, ROC/PR curve를 표시합니다.

In [ ]:
for filename in ['training_curve.png', 'threshold_sweep.png', 'window_distribution.png', 'confusion_matrix.png', 'roc_curve.png', 'pr_curve.png']:
    path = result_dir / filename
    if path.exists():
        display(Markdown(f'### {filename}'))
        display(Image(filename=str(path)))
    else:
        print('Missing:', path)

## 8. 세션 내 비교표 생성

`raw`와 `filtered`를 모두 실행한 경우 같은 세션 폴더 안에서 비교 plot을 생성합니다.

In [ ]:
run_streaming([sys.executable, 'scripts/collect_baseline_results.py', '--results-root', SESSION_OUTPUT_ROOT])
common_dir = SESSION_OUTPUT_ROOT / 'common'
summary_csv = common_dir / 'phase0_summary.csv'
if summary_csv.exists():
    display(pd.read_csv(summary_csv))
for filename in ['compare_f1_bar.png', 'compare_precision_recall.png', 'compare_auc.png', 'compare_quantization_delta.png', 'compare_model_size.png']:
    path = common_dir / filename
    if path.exists():
        display(Markdown(f'### {filename}'))
        display(Image(filename=str(path)))